# 00 — Archive QC and trace-specific noise windows

Find conservative **proposed** noise prefixes and event onsets. The initial five seconds are a working noise assumption, subject to checks for missing samples, flat padding and large transients. Extend only toward supported activity, leaving a one-second guard. Every proposal needs review; earthquake arrival times do not establish MDETECT's configured pretrigger length or absolute clock accuracy.

Source samples are unchanged. No calibration, filtering or UTC correction is applied to amplitude measurements. See `NOISE_WINDOW_STRATEGY.md` for the decision rules and limitations.

The analysis engine now lives in **FLOVOpy**: `flovopy.processing.detection.detect_pretrigger_window()`, also available as `flovopy.core.vdap.detect_pretrigger_window()`. This notebook and `pretrigger_qc.py` handle Pinatubo file selection, exports, figures and manual review. Use a kernel with the updated FLOVOpy checkout installed (the `flovopy_env` environment here).


Generated products use the shared `project_config.json` results root. Source/input files remain separate.


In [ ]:
from pathlib import Path
import sys
_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in _candidates if (p / 'project_config.json').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start in the Pinatubo repository; project_config.json must be available.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from pinatubo_project import input_path, output_path, runtime_environment
runtime_environment()

from pathlib import Path
import sys, json
_candidates = [Path.cwd(), Path.cwd() / '20_analysis', *Path.cwd().parents]
ANALYSIS_DIR = next((p for p in _candidates if (p / 'pretrigger_qc.py').is_file()), None)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('Start in 20_analysis or its project root; keep pretrigger_qc.py beside this notebook.')
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))
from flovopy.processing.detection import PretriggerSettings as Settings, detect_pretrigger_window
from flovopy.processing.event_windows import ALGORITHM_VERSION
from pretrigger_qc import select_files, analyse_files, plot_record, plot_frequency_record, plot_hilbert_trace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

from dataclasses import replace

print('FLOVOpy window engine:', ALGORITHM_VERSION)
# A loaded ObsPy record can also be analysed directly, without file outputs:
# windows = detect_pretrigger_window(st, settings=settings)
# The same implementation is discoverable as vdap.detect_pretrigger_window(st, settings=settings).


## 1. Select the waveform archive and sample

Set `WAV_ROOT_OVERRIDE` if necessary: it is the directory containing `1991/05`, `1991/06`, etc. Otherwise use the known Box archive when available, then the configured archive. An explicit override avoids probing stale external-volume paths. Selection chooses spread-out recording **days**, then a middle file on each chosen day. It does not select for clear arrivals and is not a representative statistical sample.

To focus on clear events, enter absolute waveform paths in `EXPLICIT_FILES`. Increase the sample only after checking the initial figures. Filename dates here are selection hints, not verified UTC dates.

In [ ]:
WAV_ROOT_OVERRIDE = None  # e.g. Path('/data/pinatubo/FAIR/SEISAN/WAV/PNTBO')
MONTHS = ('1991/05', '1991/06', '1991/07', '1991/08')
FILES_PER_MONTH = 1
EXPLICIT_FILES = []  # optional list of Path objects or absolute filenames
MAX_PLOT_FILES = 4
MAX_PLOT_CHANNELS = 12
OUTPUT_ROOT = output_path('20_analysis','pretrigger_qc',create=True)

_archive_candidates = [Path.home() / 'Library/CloudStorage/Box-Box/thompsong/DATA/pinatubo1991/FAIR/SEISAN/WAV/PNTBO']
_config = ANALYSIS_DIR / 'config.json'
if _config.is_file():
    _input = json.loads(_config.read_text()).get('input', {})
    if _input.get('seisan_root'):
        _archive_candidates.append(Path(_input['seisan_root']).expanduser() / 'WAV' / _input.get('database', 'PNTBO'))
WAV_ROOT = Path(WAV_ROOT_OVERRIDE).expanduser() if WAV_ROOT_OVERRIDE is not None else next((p for p in _archive_candidates if p.is_dir()), None)
if EXPLICIT_FILES:
    FILES = [Path(p).expanduser() for p in EXPLICIT_FILES]
    selection_report = pd.DataFrame({'explicit_file': [str(p) for p in FILES]})
else:
    if WAV_ROOT is None:
        raise FileNotFoundError('Set WAV_ROOT_OVERRIDE or EXPLICIT_FILES to your waveform archive.')
    FILES, selection_report = select_files(WAV_ROOT, MONTHS, FILES_PER_MONTH)
display(selection_report)
print('Waveform root:', WAV_ROOT)
print('Selected files:', len(FILES))
for path in FILES:
    print(path)
if not FILES:
    raise ValueError('No files selected; check archive and months.')


## 2. Screening settings

Use sustained strong amplitude or amplitude-supported causal STA/LTA evidence at **two distinct stations** to locate the first candidate event. Components at one station count once. Search each trace from two seconds before the cluster anchor to eight seconds after it. These adjustable bounds allow arrivals to differ; they are not travel-time calculations.

Within this interval, select the earliest sustained weak/strong amplitude or STA/LTA candidate. Weak channels without a candidate keep five seconds. Audit any extension for earlier sustained activity, strong isolated transients, missing data and incomplete bins. Ordinary isolated weak fluctuations do not automatically invalidate noise. The guard applies to the earliest limiting activity. Conflicting minimum/guard requirements yield no proposed window.

RSAM is mean absolute reference-centred counts; RMS is root mean square. Thresholds are screening heuristics, not significance tests. Hilbert and spectral changes remain independent diagnostics and do not automatically extend a window.

Onset-support columns describe the first time bin or sustained evidence window supporting the selected candidate. They are **not confidence intervals** and do not establish the exact physical start of an emergent signal. The selected onset remains the conservative start of that evidence window. Event/coda ends remain unset. For other datasets, configure the assumed initial-noise duration and station support explicitly; this notebook retains the Pinatubo defaults.


In [ ]:
settings = Settings(
    network_min_stations=2,
    network_coincidence_s=3.0,
    network_lead_s=2.0,
    network_lag_s=8.0,
    refine_amplitude_factor=1.5,
    refine_mad_factor=3.0,
    reference_start_s=0.0,
    reference_end_s=5.0,
    fine_window_s=0.25,
    amplitude_factor=3.0,
    mad_factor=6.0,
    sustain_s=0.5,
    sta_s=0.5,
    lta_s=3.0,
    sta_lta_threshold=3.0,
    min_coverage=0.95,
    plot_seconds=30.0,
    hilbert_edge_s=1.0,
    hilbert_envelope_floor_fraction=0.1,
    hilbert_smooth_s=0.25,
)
settings


## 3. Measure the pilot sample
IRIG and empty traces are excluded. Other channels are retained with QC flags; flagged candidates should be reviewed before interpretation. Masked/nonfinite gaps remain invalid. Previously zero-filled gaps cannot be reliably recovered from the converted data.

In [ ]:
result = analyse_files(FILES, OUTPUT_ROOT, settings)
OUT = result['output']
print('Output:', OUT)
display(result['files'])
display(result['network'].round(3))
qc = result['qc']
show = ['record_id', 'trace_id', 'amplitude_onset_file_s', 'sta_lta_onset_file_s',
        'selected_onset_file_s', 'selected_onset_method',
        'onset_support_start_file_s', 'onset_support_end_file_s', 'proposed_noise_end_file_s',
        'noise_proposal_status', 'noise_limiting_reason', 'qc_flags']
display(qc[[c for c in show if c in qc]].round(3))


## 4. Full records and onset zooms

Show every sampled channel, including weak channels and late secondary activity. Each file has a full-record view and a first-30-second view. Times are relative to the earliest stored non-IRIG trace. Dots show selected onset proposals; dashed per-channel lines mark proposed noise ends. Light blue marks the network search interval. Waveforms are independently scaled and clipped for display only; metrics use original centred samples.

Selected onset support is tabulated above; these intervals describe detector evidence rather than exact arrival uncertainty.


In [ ]:
for record_id, records in list(result['details'].items())[:MAX_PLOT_FILES]:
    full_seconds = max(r['file_offset_s'] + r['qc']['duration_s'] for r in records)
    for suffix, cfg in [('full', replace(settings, plot_seconds=full_seconds)), ('zoom', settings)]:
        stem = OUT / 'figures' / f'{record_id}_{suffix}'
        fig = plot_record(records, cfg, stem, max_channels=MAX_PLOT_CHANNELS)
        if fig is not None:
            display(Image(filename=str(stem.with_suffix('.png'))))
            plt.close(fig)


## Frequency cross-check

Noise proposals now use network-constrained onset refinement and the quiet-prefix audit described above. Spectral change alone is a review flag; absence of change does not prove quiet.

The first five seconds now also set the amplitude reference. The **ten-second acquisition buffer remains a plausible hypothesis**, not a confirmed setting or a forced boundary.

Three frequency diagnostics are calculated in trailing 1-s windows, every 0.25 s:

- **PSD centroid, default 0.8–20 Hz:** power-weighted average frequency; avoids the previously noted ~40-Hz artifact band, but still reflects the instrument/telemetry response.
- **Normalized spectral-shape change:** Jensen–Shannon divergence from the first-five-second reference spectrum. A pure amplitude rescaling ideally does not change it.
- **Zero-crossing proxy on raw linearly detrended samples:** both-direction crossings divided by twice the elapsed time. For a sinusoid this approximates its frequency; in broadband noise it is sensitive to small high-frequency fluctuations, quantization and acquisition noise.

The frequency candidate requires a sustained centroid shift (at least 2 Hz or the robust reference-scatter threshold) **and** a spectral-shape change. These are heuristic screening thresholds. Frequency can change upward or downward, or change shape without shifting its centroid; the current detector will not find every spectral change. Absence of a flag is not proof of quiet noise.

Plotted frequency times mark the end of each actual data window, with no endpoint padding. A candidate at `t` uses `[t−1, t)` and must not be treated as an exact arrival time; sustained confirmation also uses later windows. Overlap makes these estimates correlated. Frequency and amplitude are complementary measurements of the same samples, not statistically independent observations.

**True instantaneous frequency** is the derivative of the unwrapped Hilbert analytic-signal phase, divided by 2π. It can be useful for a dominant narrowband component, but becomes unstable near envelope zeros and ambiguous for broadband/multiple-component earthquake signals. It is now included as a separate diagnostic below; it does not determine noise-window boundaries. See [SciPy Hilbert documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.hilbert.html).

In [ ]:
frequency_fields = ['trace_id', 'selected_onset_file_s', 'frequency_change_file_s',
                    'frequency_support_start_file_s', 'proposed_noise_start_file_s',
                    'proposed_noise_end_file_s', 'noise_proposal_status', 'qc_flags']
display(qc[[c for c in frequency_fields if c in qc]].round(3))
for record_id, records in list(result['details'].items())[:MAX_PLOT_FILES]:
    fig = plot_frequency_record(records, settings, OUT / 'figures' / f'{record_id}_frequency',
                                max_channels=MAX_PLOT_CHANNELS)
    if fig is not None:
        display(Image(filename=str(OUT / 'figures' / f'{record_id}_frequency.png')))
        plt.close(fig)
print('Frequency measurements:', OUT / 'frequency_checks.csv')

## Hilbert instantaneous frequency — separate diagnostic

For each finite contiguous run, linearly detrend, construct the analytic signal with SciPy Hilbert, unwrap phase, and difference it: `f = fs * diff(phase) / (2*pi)`, located at sample midpoints. No time-domain bandpass is used.

Retain **signed** estimates, including negative values. Broadband mixed signals can have negative or unstable instantaneous frequency; removing negatives would bias summaries. Low-envelope estimates, run edges and nonfinite estimates are masked. The 0.25-second rolling median requires at least half its support and never crosses a true data gap. One-second summaries include valid coverage and the fraction of negative estimates. Interpret low-coverage summaries cautiously. This diagnostic does not pick an onset or prove a window is noise. The band-limited PSD centroid measures a different quantity.


In [ ]:
HILBERT_EXAMPLE_FILES = 2
HILBERT_TRACE_ID = None  # e.g. 'XB.UBO..EHZ'; None selects the first trace in each file
shown = 0
for record_id, records in result['details'].items():
    choices = [r for r in records if HILBERT_TRACE_ID is None or r['qc']['trace_id'] == HILBERT_TRACE_ID]
    if not choices:
        continue
    record = choices[0]
    fig = plot_hilbert_trace(record, settings, OUT / 'figures' / f'{record_id}_hilbert_detail')
    if fig is not None:
        display(Image(filename=str(OUT / 'figures' / f'{record_id}_hilbert_detail.png')))
        plt.close(fig)
        shown += 1
    if shown >= HILBERT_EXAMPLE_FILES:
        break
hilbert_columns = ['trace_id', 'hilbert_valid_fraction', 'hilbert_negative_frequency_fraction', 'hilbert_samples_path']
display(qc[[c for c in hilbert_columns if c in qc]].round(3))
print('Sample-level arrays:', OUT / 'hilbert_samples')

## 5. Selected arrival summary — exploratory only

Summarize earliest selected arrivals per file. These times include station travel-time and detection effects; a cluster near ten seconds does not prove a ten-second acquisition buffer. Check logs/metadata or known trigger times for that question. Review all proposed windows before computing event-versus-noise metrics.


In [ ]:
if 'selected_onset_file_s' in qc:
    candidates = qc.groupby('record_id')['selected_onset_file_s'].min().rename('earliest_selected_candidate_s').to_frame()
    candidates['within_1s_of_5s'] = (candidates.earliest_selected_candidate_s - 5).abs() <= 1
    candidates['within_1s_of_10s'] = (candidates.earliest_selected_candidate_s - 10).abs() <= 1
    candidates['interpretation'] = 'ARRIVAL_SCREEN_ONLY; NOT_TRIGGER_TIME'
    display(candidates.round(3))
    candidates.to_csv(OUT / 'file_candidate_summary.csv')
    valid = candidates.earliest_selected_candidate_s.dropna()
    if len(valid):
        fig, ax = plt.subplots(figsize=(9, 3.5))
        limit = max(settings.plot_seconds, float(valid.max()) + 1)
        ax.hist(valid, bins=np.arange(0, limit + 1, 1), color='#4477AA', edgecolor='white')
        for hypothesis in (5, 10):
            ax.axvline(hypothesis, color='black', ls='--', lw=1, label=f'{hypothesis}-s hypothesis')
        ax.set(xlabel='Earliest selected arrival after file start (s)', ylabel='Sample files',
               title='Exploratory arrival timing; buffer length is not inferred')
        ax.legend(frameon=False)
        fig.tight_layout()
        for extension in ('png', 'pdf', 'svg'):
            fig.savefig(OUT / f'candidate_onset_histogram.{extension}', dpi=180)
        display(Image(filename=str(OUT / 'candidate_onset_histogram.png')))
        plt.close(fig)
else:
    print('No candidate columns: review file/trace errors above.')


## 6. Record manual decisions before using any noise/event metrics

`manual_window_review.csv` is a review template, **not automatically accepted windows**. Enter seconds relative to the file reference shown in the plots. Leave uncertain rows unreviewed. Keep intervals away from padding, gaps and ambiguous onsets. The initial reference interval is not prefilled as valid noise.

Suggested review statuses: `ACCEPTED`, `REJECTED`, `UNRESOLVED`. Document why each accepted interval is quiet or contains the intended signal. This notebook preserves an existing review file when rerun with the same inputs/settings. A new input/settings fingerprint creates a separate output directory. Fingerprints include file paths, sizes and modification times, not waveform-content hashes.

Changing algorithm settings may alter candidates; numerical features and production StationXML are outside this pilot.

In [ ]:
review_path = OUT / 'manual_window_review.csv'
print('Edit this table after examining the figures:', review_path)
display(pd.read_csv(review_path).head(20))
print('Tables and run manifest:', OUT)
print('Source waveforms were not modified.')
